In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp
import ipywidgets
import plotly.graph_objects as go
import rasterio
from matplotlib.colors import LightSource
from ipywidgets import interact, FloatSlider, IntSlider
import plotly.io as pio
from scipy.spatial import Delaunay
import pyvista as pv
import rasterio
from scipy.interpolate import RegularGridInterpolator
from plotly.subplots import make_subplots
from rasterio.enums import Resampling
from rasterio.windows import Window
pio.renderers.default = 'vscode'
from utils import *
import trimesh
import pyvista as pv

In [ ]:
file_path = "/Users/pierazhi/Desktop/tifs/LDEM_20M.tif" # Loads .tif
dimension_scene, _ = read_tif(file_path) # Gets dimension of the scene
#visualize_tif(file_path, np.floor(dimension_scene / 4)) # Shows half the scene (cause it is big)

In [ ]:
file_cut = '/Users/pierazhi/Desktop/tifs/LDEM_20M_clip.tif' # Creates the cut file's path
cut_pixels = 150 # Choose the square, starting from the center, that is going to be cut from the whole scene

# write_and_save_tif(file_path, file_cut, cut_pixels, 47, 43) # Saves the cut scene
# write_and_save_tif(file_path, file_cut, cut_pixels, 0, 0) # Saves the cut scene
write_and_save_tif(file_path, file_cut, cut_pixels, -25, 25) # Saves the cut scene

dimension_scene_cut, _ = read_tif(file_cut) # Read the cut scene
visualize_tif(file_cut, dimension_scene_cut) # Viualizes it

In [ ]:
passo = int(cut_pixels)
X, Y, Z, triangoli, Xe, Ye, Ze, normali, baricentri, Xg, Yg, Zg = tif2mesh(file_cut, np.floor(dimension_scene_cut*3), passo) # Creates the mesh of triangles from the starting .tif

nuova_risoluzione_metri = 5
file_cut_up = f'/Users/pierazhi/Desktop/tifs/LDEM_{nuova_risoluzione_metri}M_clip_up.tif' # Creates file's path for the cut and upsampled scene 
passo_up = cut_pixels * 2
upsampling_geotiff(file_cut, file_cut_up, nuova_risoluzione_metri, metodo_interpolazione='cubic') # Upsamples the cut file to a new resolution
X_up, Y_up, Z_up, triangoli_up, Xe_up, Ye_up, Ze_up, normali_up, baricentri_up, Xg_up, Yg_up, Zg_up  = tif2mesh(file_cut_up, np.floor(dimension_scene_cut*3), passo_up) # Creates the mesh of triangles from the starting .tif

In [ ]:
show_normals = False
show_triangles = False

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=(
        f"Mesh Originale (20m) - Triangoli: {len(triangoli):,}", 
        f"Mesh Interpolata {nuova_risoluzione_metri} m - Triangoli: {len(triangoli_up):,}"
    )
)
fig.add_trace(go.Mesh3d(
    x=X, y=Y, z=Z, i=triangoli[:, 0], j=triangoli[:, 1], k=triangoli[:, 2],
    intensity=Z, colorscale='Earth_r', showscale=False, opacity=0.85, name="Orig 20m"
), row=1, col=1)

if show_triangles:
    fig.add_trace(go.Scatter3d(
        x=Xe, y=Ye, z=Ze, mode='lines',
        line=dict(color='rgb(30,30,30)', width=1), showlegend=False
    ), row=1, col=1)

if show_normals:
    fig.add_trace(go.Scatter3d(
        x=Xg, y=Yg, z=Zg, mode='lines',
        line=dict(color='rgb(255, 0, 0)', width=3), name="Vettori Normali", showlegend=False
    ), row=1, col=1)

fig.add_trace(go.Mesh3d(
    x=X_up, y=Y_up, z=Z_up, i=triangoli_up[:, 0], j=triangoli_up[:, 1], k=triangoli_up[:, 2],
    intensity=Z_up, colorscale='Earth_r', showscale=True, opacity=0.85,
    colorbar=dict(title="Quota (m)", x=1.05), name="Interp 10m"
), row=1, col=2)

if show_triangles:
    fig.add_trace(go.Scatter3d(
        x=Xe_up, y=Ye_up, z=Ze_up, mode='lines',
        line=dict(color='rgb(50,50,50)', width=0.8), showlegend=False
    ), row=1, col=2)

if show_normals:
    fig.add_trace(go.Scatter3d(
        x=Xg_up, y=Yg_up, z=Zg_up, mode='lines',
        line=dict(color='rgb(255, 0, 0)', width=3), name="Vettori Normali", showlegend=False
    ), row=1, col=2)

scene_config = dict(
    xaxis_title="X (Metri)",
    yaxis_title="Y (Metri)",
    zaxis_title="Z (Metri)",
    aspectmode='data' 
    # aspectmode='manual',
    # aspectratio=dict(x=1, y=1, z=0.4)
)

fig.update_layout(
    scene=scene_config,   # Configura la vista di sinistra (scena 1)
    scene2=scene_config,  # Configura la vista di destra (scena 2)
    margin=dict(l=0, r=50, b=0, t=110),
    width=1150,           # Allargato leggermente per fare spazio alla barra dei colori
    height=750
)
fig.show()


In [ ]:
# --- 1. CONFIGURAZIONE MESH ---
vertices = np.vstack((X_up, Y_up, Z_up)).T
mesh_trimesh = trimesh.Trimesh(vertices=vertices, faces=triangoli_up)
mesh_trimesh.fix_normals()

faces_pv = np.hstack(np.pad(triangoli_up, ((0, 0), (1, 0)), constant_values=3))
mesh_pv = pv.PolyData(mesh_trimesh.vertices, faces_pv)

# --- 2. CONFIGURAZIONE INIZIALE FINESTRA E TERRENO ---
plotter = pv.Plotter()
plotter.add_mesh(mesh_pv, scalars=mesh_pv.points[:, 2], cmap="terrain", lighting=True, opacity=0.8)

# Calcolo dei confini del terreno (matrice 2x3)
xmin, ymin, zmin = mesh_trimesh.bounds[0]
xmax, ymax, zmax = mesh_trimesh.bounds[1]
offset_pos = 1000  

# Dizionario di stato UNIFICATO per tutti i parametri modificabili
stato_parametri = {
    "xx": np.mean(X_up) - 100,
    "yy": np.mean(Y_up) - 100,
    "zz": np.max(Z_up) + 200,
    "azimut": 135.0,
    "elevazione": -45.0
}

# Lista globale dinamica per tracciare gli attori grafici e l'attore del testo
attori_raggi = []
attore_testo = None  # Tiene traccia del pannello di testo a schermo

MAX_BOUNCES = 4
colori_raggi = ["red", "yellow", "magenta", "black"]

# --- 3. FUNZIONE DI AGGIORNAMENTO GEOMETRIA E TESTO (CALLBACK) ---
def aggiorna_ray_tracing(*args):
    """Ricalcola ricorsivamente fino a 4 rimbalzi e aggiorna il testo dinamico a schermo."""
    global attori_raggi, attore_testo
    
    # 1. Rimuove tutti i vettori e i punti della passata precedente dalla GPU
    for attore in attori_raggi:
        plotter.remove_actor(attore)
    attori_raggi.clear()
    
    # Rimuove il vecchio blocco di testo sovrimpresso se esiste
    if attore_testo is not None:
        plotter.remove_actor(attore_testo)

    # 2. Inizializzazione variabili leggendo i dati aggiornati dagli slider
    origine_corrente = np.array([stato_parametri["xx"], stato_parametri["yy"], stato_parametri["zz"]])
    direzione_corrente = calcola_direzione_raggio(stato_parametri["azimut"], stato_parametri["elevazione"])
    
    # Costruiamo la stringa informativa da mostrare direttamente sullo schermo 3D
    stringa_log_schermo = (
        f"SORGENTE TX\n"
        f"Pos: [{stato_parametri['xx']:.1f}, {stato_parametri['yy']:.1f}, {stato_parametri['zz']:.1f}]\n"
        f"Angoli: Az={stato_parametri['azimut']:.1f}°, El={stato_parametri['elevazione']:.1f}°\n"
        f"--------------------------------------------------\n"
    )

    # Disegna la sorgente iniziale (TX) come una sfera blu
    tx_sfera = pv.Sphere(radius=25.0, center=origine_corrente)
    attori_raggi.append(plotter.add_mesh(tx_sfera, color="blue", lighting=True))

    # 3. Esecuzione dei rimbalzi (fino a un massimo di 4)
    for bounce in range(MAX_BOUNCES):
        p_end = origine_corrente + (direzione_corrente * 10000.0)
        p_intersezioni, id_facce = mesh_pv.ray_trace(origine_corrente, p_end)

        if len(p_intersezioni) > 0:
            # Ordinamento spaziale per distanza
            distanze = np.linalg.norm(p_intersezioni - origine_corrente, axis=1)
            idx_vicino = np.argmin(distanze)
            
            punto_contatto = p_intersezioni[idx_vicino, :]
            
            # Estrazione sicura dell'indice triangolo
            if hasattr(id_facce, "__len__"):
                if len(id_facce) > 0:
                    idx_faccia = int(id_facce[idx_vicino])
                else:
                    idx_faccia = int(id_facce)
            else:
                idx_faccia = int(id_facce)

            # Aggiunge i dati del rimbalzo corrente alla stringa video
            stringa_log_schermo += f"Rimbalzo {bounce+1}: Triangolo {idx_faccia} -> XYZ: {np.round(punto_contatto, 1)}\n"

            # Disegna il raggio corrente e la sfera nel punto di impatto
            attori_raggi.append(plotter.add_mesh(pv.Line(origine_corrente, punto_contatto), color=colori_raggi[bounce], line_width=4))
            attori_raggi.append(plotter.add_mesh(pv.PolyData(punto_contatto), color=colori_raggi[bounce], point_size=12, render_points_as_spheres=True))

            # Calcolo della riflessione specchiata tramite normali Trimesh
            normale_grezza = mesh_trimesh.face_normals[idx_faccia].copy()
            normale_grezza /= np.linalg.norm(normale_grezza)
            
            normale_superficie = -normale_grezza if np.dot(direzione_corrente, normale_grezza) > 0 else normale_grezza
            
            direzione_riflessa = direzione_corrente - 2 * np.dot(direzione_corrente, normale_superficie) * normale_superficie
            direzione_riflessa /= np.linalg.norm(direzione_riflessa)

            # Aggiornamento parametri per il ciclo successivo con offset epsilon centimetrico
            direzione_corrente = direzione_riflessa
            origine_corrente = punto_contatto + (normale_superficie * 1e-2)
        else:
            # Il raggio si disperde nel cielo aperto
            punto_fine_disperso = origine_corrente + direzione_corrente * 1000.0
            attori_raggi.append(plotter.add_mesh(pv.Line(origine_corrente, punto_fine_disperso), color=colori_raggi[bounce], line_width=2))
            stringa_log_schermo += f"Rimbalzo {bounce+1}: Disperso nello spazio.\n"
            break

    # 4. AGGIORNA IL PANNELLO DI TESTO SOVRIMPRESSO IN ALTO A DESTRA (In tempo reale)
    attore_testo = plotter.add_text(
        stringa_log_schermo, 
        position="upper_right", 
        font_size=10, 
        color="white", 
        font="courier",
        shadow=True
    )


# --- 4. FUNZIONI DI CALLBACK PER AGGIORNARE LO STATO ---
def cb_xx(val): stato_parametri["xx"] = val; aggiorna_ray_tracing()
def cb_yy(val): stato_parametri["yy"] = val; aggiorna_ray_tracing()
def cb_zz(val): stato_parametri["zz"] = val; aggiorna_ray_tracing()
def cb_az(val): stato_parametri["azimut"] = val; aggiorna_ray_tracing()
def cb_el(val): stato_parametri["elevazione"] = val; aggiorna_ray_tracing()


# --- 5. CREAZIONE DEI 5 SLIDER INTERATTIVI ---
plotter.add_slider_widget(callback=cb_xx, rng=[xmin-offset_pos, xmax+offset_pos], value=stato_parametri["xx"], title="Posizione X Sorgente", pointa=(0.05, 0.92), pointb=(0.30, 0.92), style="modern", color="cyan")
plotter.add_slider_widget(callback=cb_yy, rng=[ymin-offset_pos, ymax+offset_pos], value=stato_parametri["yy"], title="Posizione Y Sorgente", pointa=(0.05, 0.81), pointb=(0.30, 0.81), style="modern", color="cyan")
plotter.add_slider_widget(callback=cb_zz, rng=[zmin, zmax+offset_pos], value=stato_parametri["zz"], title="Posizione Z (Altezza)", pointa=(0.05, 0.70), pointb=(0.30, 0.70), style="modern", color="cyan")
plotter.add_slider_widget(callback=cb_az, rng=[0.0, 360.0], value=stato_parametri["azimut"], title="Azimut (Gradi)", pointa=(0.05, 0.59), pointb=(0.30, 0.59), style="modern", color="cyan")
plotter.add_slider_widget(callback=cb_el, rng=[-90.0, 90.0], value=stato_parametri["elevazione"], title="Elevazione (Gradi)", pointa=(0.05, 0.48), pointb=(0.30, 0.48), style="modern", color="cyan")

# Esecuzione del calcolo iniziale al boot
aggiorna_ray_tracing()

# Avvia la visualizzazione grafica
plotter.show()


In [ ]:
# # Generazione della mesh (Nota: Z_up ripristinato senza segno meno come da tua ultima richiesta)
# vertices = np.vstack((X_up, Y_up, Z_up)).T
# mesh_luna = trimesh.Trimesh(vertices=vertices, faces=triangoli_up)
# mesh_luna.fix_normals()

# xmin, ymin, zmin = mesh_luna.bounds[0]
# xmax, ymax, zmax = mesh_luna.bounds[1]

# # Calcolo del centro geometrico della mesh
# centro_mesh = mesh_luna.centroid

# # Creazione degli assi cartesiani centrati sulla mesh (dimensione scalata sui bound)
# dimensione_assi = max(xmax - xmin, ymax - ymin, zmax - zmin) * 0.2
# matrice_centro = np.eye(4)
# matrice_centro[:3, 3] = centro_mesh

# # Creazione degli assi cartesiani posizionati nel centro geometrico
# assi_riferimento = trimesh.creation.axis(
#     transform=matrice_centro, 
#     axis_length=dimensione_assi, 
#     axis_radius=dimensione_assi * 0.01
# )
# # Configurazione del colore iniziale della mesh per evitare sovrascritture continue
# colore_uniforme = np.tile([200, 200, 200, 150], (len(mesh_luna.faces), 1))
# mesh_luna.visual.face_colors = colore_uniforme

# print(f"Mesh caricata e normalizzata correttamente!")
# print(f"Numero di vertici: {len(mesh_luna.vertices)}")
# print(f"Numero di triangoli: {len(mesh_luna.faces)}")
# print(f"Centro della mesh calcolato in: {centro_mesh}")

# offset_pos = 1000
# intersector = RayMeshIntersector(mesh_luna)

# def test(xx, yy, zz, az, el):
#     tx_fittizio = np.array([xx, yy, zz])
#     direzione_corrente = calcola_direzione_raggio(az, el)
#     origine_corrente = tx_fittizio.copy()
    
#     # Lista elementi geometrici inclusi la mesh e gli assi cartesiani permanenti
#     geometriche_scena = [mesh_luna, assi_riferimento]
    
#     # Inseriamo il TX (Sfera Rossa)
#     tx_sfera = trimesh.creation.icosphere(subdivisions=2, radius=50.0) 
#     tx_sfera.apply_translation(tx_fittizio)
#     tx_sfera.visual.face_colors = [255, 0, 0, 255]
#     geometriche_scena.append(tx_sfera)
    
#     MAX_BOUNCES = 4
#     colori_raggi = [
#         [255, 165, 0, 255],
#         [0, 255, 0, 255],
#         [255, 0, 255, 255],
#         [0, 255, 255, 255]
#     ]
    
#     for bounce in range(MAX_BOUNCES):
#         punti_impatto, index_ray, index_tri = intersector.intersects_location(
#             ray_origins=[origine_corrente], 
#             ray_directions=[direzione_corrente]
#         )
        
#         if len(punti_impatto) == 0:
#             punto_fine = origine_corrente + direzione_corrente * 500.0
#             crea_segmento_raggio(origine_corrente, punto_fine, colori_raggi[bounce], geometriche_scena)
#             print(f"Rimbalzo {bounce+1}: Il raggio si è disperso nello spazio.")
#             break
            
#         punto_contatto = punti_impatto[0]
#         idx_faccia = index_tri[0]
        
#         crea_segmento_raggio(origine_corrente, punto_contatto, colori_raggi[bounce], geometriche_scena)
        
#         sfera_impatto = trimesh.creation.icosphere(subdivisions=2, radius=12.0 - (bounce * 2)) 
#         sfera_impatto.apply_translation(punto_contatto)
#         sfera_impatto.visual.face_colors = colori_raggi[bounce]
#         geometriche_scena.append(sfera_impatto)
        
#         print(f"Rimbalzo {bounce+1}: Impatto sulla faccia {idx_faccia} alle coordinate {punto_contatto}")
        
#         normale_grezza = mesh_luna.face_normals[idx_faccia]
#         prodotto_scalare = np.dot(direzione_corrente, normale_grezza)
#         normale_superficie = -normale_grezza if prodotto_scalare > 0 else normale_grezza
        
#         direzione_riflessa = direzione_corrente - 2 * np.dot(direzione_corrente, normale_superficie) * normale_superficie
#         direzione_riflessa /= np.linalg.norm(direzione_riflessa)
        
#         direzione_corrente = direzione_riflessa
#         origine_corrente = punto_contatto + direzione_corrente * 1.0

#     # Inizializzazione della scena fuori dal ciclo for
#     scena = trimesh.Scene(geometriche_scena)
    
#     # Configurazione della luce ambientale (Risoluzione aree nere)
#     scena.lights.clear()
    
        
#     return scena.show(viewer='notebook')

# def crea_segmento_raggio(inizio, fine, colore, lista_scena):
#     vettore = fine - inizio
#     lunghezza = np.linalg.norm(vettore)
#     if lunghezza < 0.001: return
    
#     cilindro = trimesh.creation.cylinder(radius=5.0, height=lunghezza)
#     posizione_media = (inizio + fine) / 2
#     matrice = trimesh.geometry.align_vectors([0, 0, 1], vettore / lunghezza)
#     matrice[:3, 3] = posizione_media
#     cilindro.apply_transform(matrice)
#     cilindro.visual.face_colors = colore
#     lista_scena.append(cilindro)

# # Esecuzione dell'interfaccia interattiva
# ipywidgets.interact(test, 
#     xx=ipywidgets.FloatSlider(min=xmin-offset_pos, max=xmax+offset_pos, step=10, value=(xmin+xmax)/2, description='Pos X | TX'),
#     yy=ipywidgets.FloatSlider(min=ymin-offset_pos, max=ymax+offset_pos, step=10, value=(ymin+ymax)/2, description='Pos Y | TX'),
#     zz=ipywidgets.FloatSlider(min=zmin, max=zmax+offset_pos, step=10, value=zmax+100.0, description='Pos Z | TX'),
#     az=ipywidgets.IntSlider(min=1, max=360, step=2, value=93, description='Azimuth'),
#     el=ipywidgets.IntSlider(min=-90, max=90, step=1, value=-45, description='Elevation'),
# )


In [ ]:
# # 1. Creazione e inizializzazione della mesh (Eseguito una sola volta)
# vertices = np.vstack((X_up, Y_up, -Z_up)).T
# mesh_luna = trimesh.Trimesh(vertices=vertices, faces=triangoli_up)
# mesh_luna.fix_normals()

# xmin, ymin, zmin = mesh_luna.bounds[0]
# xmax, ymax, zmax = mesh_luna.bounds[1]

# # CORREZIONE: Calcolo globale delle coordinate centrali del terreno
# centro_x = (xmin + xmax) / 2
# centro_y = (ymin + ymax) / 2

# # Configurazione del colore iniziale della mesh per evitare sovrascritture continue
# colore_uniforme = np.tile([200, 200, 200, 150], (len(mesh_luna.faces), 1))
# mesh_luna.visual.face_colors = colore_uniforme

# print(f"Mesh caricata e normalizzata correttamente!")
# print(f"Numero di vertici: {len(mesh_luna.vertices)}")
# print(f"Numero di triangoli_up: {len(mesh_luna.faces)}")

# offset_pos = 1000

# intersector = RayMeshIntersector(mesh_luna)

# # =====================================================================
# # FUNZIONE DI SUPPORTO PER GENERARE LA GRIGLIA DI TX IN ARIA SPAZIATA
# # =====================================================================
# def genera_griglia_tx_spaziata(num_tx_lato, spaziatura_metri, altitudine_assoluta):
#     """
#     Genera una griglia regolare di TX simmetrica rispetto al centro del terreno,
#     utilizzando una spaziatura fissa in metri definita dall'utente.
#     """
#     if num_tx_lato == 1:
#         return np.array([[centro_x, centro_y, altitudine_assoluta]])
        
#     # Calcola l'estensione totale occupata dalla griglia su un asse
#     estensione_asse = (num_tx_lato - 1) * spaziatura_metri
    
#     # Determina i punti di partenza e arrivo centrati rispetto al centro del DEM
#     inizio_x = centro_x - (estensione_asse / 2)
#     inizio_y = centro_y - (estensione_asse / 2)
    
#     # Genera le coordinate X e Y spaziate esattamente dei metri richiesti
#     x_coords = np.arange(num_tx_lato) * spaziatura_metri + inizio_x
#     y_coords = np.arange(num_tx_lato) * spaziatura_metri + inizio_y
    
#     X, Y = np.meshgrid(x_coords, y_coords)
#     punti_2d = np.vstack([X.ravel(), Y.ravel()]).T
    
#     # Assegna la quota Z costante a tutta la flotta aerea
#     quote_z = np.ones((len(punti_2d), 1)) * altitudine_assoluta
#     return np.hstack([punti_2d, quote_z])

# def test(n_tx_lato, spaziatura_tx, altezza_flotta, az, el):
#     direzione_iniziale = calcola_direzione_raggio(az, el)
#     geometriche_scena = [mesh_luna]
    
#     # Genera la griglia planare di TX applicando la spaziatura in tempo reale
#     quota_assoluta_volo = zmax + altezza_flotta
#     lista_trasmettitori = genera_griglia_tx_spaziata(n_tx_lato, spaziatura_tx, quota_assoluta_volo)
    
#     MAX_BOUNCES = 4
#     colori_raggi = [
#         [255, 165, 0, 255],   # 1° impatto (Arancione)
#         [0, 255, 0, 255],     # 2° impatto (Verde)
#         [255, 0, 255, 255],   # 3° impatto (Viola)
#         [0, 255, 255, 255]    # 4° impatto (Ciano)
#     ]
    
#     for idx_tx, tx_fittizio in enumerate(lista_trasmettitori):
#         tx_sfera = trimesh.creation.icosphere(subdivisions=2, radius=50.0) 
#         tx_sfera.apply_translation(tx_fittizio)
#         tx_sfera.visual.face_colors = [255, 0, 0, 255]
#         geometriche_scena.append(tx_sfera)
        
#         origine_corrente = tx_fittizio.copy()
#         direzione_corrente = direzione_iniziale.copy()
        
#         for bounce in range(MAX_BOUNCES):
#             punti_impatto, index_ray, index_tri = intersector.intersects_location(
#                 ray_origins=[origine_corrente], 
#                 ray_directions=[direzione_corrente]
#             )
            
#             if len(punti_impatto) == 0:
#                 punto_fine = origine_corrente + direzione_corrente * 500.0
#                 crea_segmento_raggio(origine_corrente, punto_fine, colori_raggi[bounce], geometriche_scena)
#                 print(f"TX #{idx_tx+1} - Rimbalzo {bounce+1}: Il raggio si è disperso nello spazio.")
#                 break
                
#             punto_contatto = punti_impatto[0]
#             idx_faccia = index_tri[0]
            
#             crea_segmento_raggio(origine_corrente, punto_contatto, colori_raggi[bounce], geometriche_scena)
            
#             sfera_impatto = trimesh.creation.icosphere(subdivisions=2, radius=12.0 - (bounce * 2)) 
#             sfera_impatto.apply_translation(punto_contatto)
#             sfera_impatto.visual.face_colors = colori_raggi[bounce]
#             geometriche_scena.append(sfera_impatto)
            
#             print(f"TX #{idx_tx+1} - Rimbalzo {bounce+1}: Impatto sulla faccia {idx_faccia} alle coordinate {punto_contatto}")
            
#             normale_grezza = mesh_luna.face_normals[idx_faccia]
#             prodotto_scalare = np.dot(direzione_corrente, normale_grezza)
#             normale_superficie = -normale_grezza if prodotto_scalare > 0 else normale_grezza
            
#             direzione_riflessa = direzione_corrente - 2 * np.dot(direzione_corrente, normale_superficie) * normale_superficie
#             direzione_riflessa /= np.linalg.norm(direzione_riflessa)
            
#             direzione_corrente = direzione_riflessa
#             origine_corrente = punto_contatto + direzione_corrente * 1.0

#     scena = trimesh.Scene(geometriche_scena)
    
#     # Configurazione telecamera fissa per prevenire il super-zoom
#     raggio_mappa = scena.scale
#     centro_mappa = scena.centroid
#     posizione_camera = centro_mappa + np.array([raggio_mappa * 0.9, raggio_mappa * 0.9, raggio_mappa * 1.4])
    
#     asse_z = posizione_camera - centro_mappa
#     asse_z /= np.linalg.norm(asse_z)
#     asse_x = np.cross(np.array([0.0, 0.0, 1.0]), asse_z)
#     asse_x /= np.linalg.norm(asse_x)
#     asse_y = np.cross(asse_z, asse_x)
    
#     matrice_telecamera = np.eye(4)
#     matrice_telecamera[:3, 0] = asse_x
#     matrice_telecamera[:3, 1] = asse_y
#     matrice_telecamera[:3, 2] = asse_z
#     matrice_telecamera[:3, 3] = posizione_camera
#     scena.camera_transform = matrice_telecamera
    
#     return scena.show(viewer='notebook')

# # Funzione di supporto per generare i cilindri tridimensionali dei raggi
# def crea_segmento_raggio(inizio, fine, colore, lista_scena):
#     vettore = fine - inizio
#     lunghezza = np.linalg.norm(vettore)
#     if lunghezza < 0.001: return
    
#     cilindro = trimesh.creation.cylinder(radius=5.0, height=lunghezza)
#     posizione_media = (inizio + fine) / 2
#     matrice = trimesh.geometry.align_vectors([0, 0, 1], vettore / lunghezza)
#     matrice[:3, 3] = posizione_media
#     cilindro.apply_transform(matrice)
#     cilindro.visual.face_colors = colore
#     lista_scena.append(cilindro)

# # CORREZIONE: Aggiunto lo slider 'spaziatura_tx' all'interno di ipywidgets.interact
# ipywidgets.interact(test, 
#     n_tx_lato=ipywidgets.IntSlider(min=1, max=10, step=1, value=2, description='TX per lato'),
#     spaziatura_tx=ipywidgets.FloatSlider(min=10.0, max=500.0, step=10, value=150.0, description='Spaziatura TX (m)'),
#     altezza_flotta=ipywidgets.FloatSlider(min=50.0, max=1000.0, step=50, value=200.0, description='Altezza TX'),
#     az=ipywidgets.IntSlider(min=1, max=360, step=2, value=93, description='Azimuth'),
#     el=ipywidgets.IntSlider(min=-90, max=90, step=1, value=-45, description='Elevation'),
# )

# print("ORA???")